# Solutions · 17 · Constrained optimisation and linear programming

Worked solutions to the exercises of [notebook 17](../notebooks/17_optimisation_constraints_lp.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engmath is missing): install it from GitHub
    import engmath
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engmath"], check=True)
import numpy as np
import matplotlib.pyplot as plt
import engmath
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")
from engmath import optimize
from scipy.optimize import linprog, milp, LinearConstraint, Bounds, minimize
c = [400, 300]
A = [[2, 1], [1, 1], [1, 0]]
b = [100, 80, 40]
base = optimize.simplex_lp(c, A, b)

## Exercise 1

The workshop can buy up to 10 extra assembly hours at EUR 150 per hour. Should it? Use the shadow
   price, then confirm by re-solving.

In [2]:
print(f"shadow price of assembly: EUR {base.shadow_prices[1]:.0f} per hour; cost EUR 150 per hour")
for extra in (0, 5, 10, 20, 25):
    lp = optimize.simplex_lp(c, A, [100, 80 + extra, 40])
    print(f"+{extra:>2} h: plan {np.round(lp.x, 1)}, profit EUR {lp.objective:,.0f} "
          f"(+{lp.objective - base.objective:,.0f}; minus overtime cost {lp.objective - base.objective - 150*extra:+,.0f})")

shadow price of assembly: EUR 200 per hour; cost EUR 150 per hour
+ 0 h: plan [20. 60.], profit EUR 26,000 (+0; minus overtime cost +0)
+ 5 h: plan [15. 70.], profit EUR 27,000 (+1,000; minus overtime cost +250)
+10 h: plan [10. 80.], profit EUR 28,000 (+2,000; minus overtime cost +500)
+20 h: plan [  0. 100.], profit EUR 30,000 (+4,000; minus overtime cost +1,000)
+25 h: plan [  0. 100.], profit EUR 30,000 (+4,000; minus overtime cost +250)


Each extra assembly hour earns EUR 200 and costs EUR 150, so buying all 10 hours adds EUR 500 net. The
shadow price is valid only while the same constraints stay binding: the plan shifts towards heavy-duty
pumps, and after 20 extra hours no standard pumps are left - beyond that, another hour is worth less.
Re-solving confirms the range in which the shadow price holds.

## Exercise 2

**Whole pumps.** A redesigned standard pump needs 2.4 h of machining, and 105 machining hours are
   available. Solve the LP, then round its solution to whole pumps (to the nearest integer, and
   downwards). Are the rounded plans feasible? Optimal? Compare with `scipy.optimize.milp`, which
   requires whole numbers.

In [3]:
A2 = np.array([[2.4, 1], [1, 1], [1, 0]]); b2 = [105, 80, 40]
lp = linprog([-400, -300], A_ub=A2, b_ub=b2, method="highs")
print(f"LP optimum: {np.round(lp.x, 3)}, profit EUR {-lp.fun:,.1f}")
for label, plan in (("rounded to nearest", np.round(lp.x)), ("rounded down", np.floor(lp.x))):
    feasible = np.all(A2 @ plan <= np.array(b2) + 1e-9)
    print(f"{label:<19}: {plan} feasible: {feasible}" + (f", profit EUR {400*plan[0] + 300*plan[1]:,.0f}" if feasible else
          f" (machining {A2[0] @ plan:.1f} h > 105 h)"))
ip = milp([-400, -300], constraints=LinearConstraint(A2, -np.inf, b2), integrality=[1, 1], bounds=Bounds(0, np.inf))
print(f"integer optimum (milp): {ip.x}, profit EUR {-ip.fun:,.0f}")

LP optimum: [17.857 62.143], profit EUR 25,785.7
rounded to nearest : [18. 62.] feasible: False (machining 105.2 h > 105 h)
rounded down       : [17. 62.] feasible: True, profit EUR 25,400
integer optimum (milp): [17. 63.], profit EUR 25,700


Rounding to the nearest integers **violates** the machining limit, and rounding down is feasible but gives
up EUR 300 of profit compared with the true integer optimum (17, 63), which neither rounding finds.
Integer programs need integer methods (branch and bound, as in `milp`), especially when quantities are
small; for large quantities rounding is usually harmless.

## Exercise 3

The tank's end caps must be twice as thick as its wall. Minimise the material *volume*
   ($2\cdot 2\pi r^2 t + 2\pi r h t$). How does the optimal $h/r$ change? Derive it with SymPy.

In [4]:
import sympy as sp
r, h, lam, V = sp.symbols("r h lambda V", positive=True)
material = 2*(2*sp.pi*r**2) + 2*sp.pi*r*h            # caps twice as thick (wall thickness t factored out)
sol = sp.solve([sp.diff(material - lam*(sp.pi*r**2*h - V), v) for v in (r, h, lam)], [r, h, lam], dict=True)[0]
print("h/r =", sp.simplify(sol[h]/sol[r]))
num = minimize(lambda v: 4*np.pi*v[0]**2 + 2*np.pi*v[0]*v[1], [0.5, 1.0], method="SLSQP",
               bounds=[(1e-3, None)]*2, constraints=[{"type": "eq", "fun": lambda v: np.pi*v[0]**2*v[1] - 1}])
print(f"numerical check: r = {num.x[0]:.4f} m, h = {num.x[1]:.4f} m, h/r = {num.x[1]/num.x[0]:.3f}")

h/r = 4
numerical check: r = 0.4301 m, h = 1.7205 m, h/r = 4.000


With end caps twice as thick, material in the caps is more expensive, so the optimal tank becomes taller
and slimmer: $h = 4r$ instead of $h = 2r$. In general, if the caps cost $c$ times as much per unit area as
the wall, $h/r = 2c$ - a result worth deriving once symbolically rather than re-optimising every case.

## Exercise 4

**Implement it yourself:** the *log-barrier* (interior-point) method minimises $f(x) - \mu\sum\ln(-g_i(x))$ for decreasing $\mu$, staying strictly feasible. Implement it with `optimize.nelder_mead` for the tank with $h \le 1$ (eliminate the volume constraint first) and compare with the penalty method.

In [5]:
area_r = lambda r_: 2*np.pi*r_**2 + 2/r_            # volume constraint eliminated: h = 1/(pi r^2)
g = lambda r_: 1/(np.pi*r_**2) - 1.0                # h <= 1  <=>  g(r) <= 0
r_cur = 1.0                                          # a strictly feasible start (h = 0.32 m)
for mu in 10.0**-np.arange(0, 9):
    barrier = lambda v, mu=mu: area_r(v[0]) - mu*np.log(-g(v[0])) if g(v[0]) < 0 else np.inf
    r_cur = optimize.nelder_mead(barrier, [r_cur], step=0.05, tol=1e-14).x[0]
    if mu in (1.0, 1e-3, 1e-8):
        print(f"mu = {mu:.0e}: r = {r_cur:.8f} m, h = {1/(np.pi*r_cur**2):.8f} m")
print(f"exact: r = {np.sqrt(1/np.pi):.8f} m, h = 1")

mu = 1e+00: r = 0.70811210 m, h = 0.63481342 m
mu = 1e-03: r = 0.56536580 m, h = 0.99584341 m
mu = 1e-08: r = 0.56418960 m, h = 0.99999996 m
exact: r = 0.56418958 m, h = 1


The barrier $-\mu\ln(-g)$ grows to infinity at the constraint boundary, so every iterate stays strictly
feasible (the tank is never taller than 1 m), and the optimum is approached from *inside* as $\mu \to 0$. The
penalty method approaches from *outside*, through infeasible designs. Interior-point methods built on
this idea are the workhorses of modern large-scale optimisation. Eliminating the equality constraint
first (using $h = 1/(\pi r^2)$) turned the problem into a one-variable one.